<a href="https://colab.research.google.com/github/jceltruda/Projects-in-AI-and-ML/blob/main/Project_6/ML_AI_Projects_6_Task_1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Part 1

### FrozenLake MDP:
* State Space: 16 states representing a location on a 4x4 grid of a FrozenLake with is_slippery=True.
* Action Space: 4 actions representing a step to take (0: Left, 1: Down, 2: Right, 3: Up).
* Reward Structure: +1 Reward granted for reaching the goal (bottom right corner of grid), 0 otherwise.
* Discount Factor: 0.99, making the agent value long-term rewards.
* Experiment: Lowering Discount Factor to 0.1

In [1]:
import gymnasium as gym
import numpy as np

env = gym.make('FrozenLake-v1', is_slippery=True)

def value_iteration(env, gamma=0.99, theta=1e-8):
    n_states = env.observation_space.n
    n_actions = env.action_space.n
    V = np.zeros(n_states)
    policy = np.zeros(n_states, dtype=int)

    # Value Iteration
    while True:
        delta = 0
        for s in range(n_states):
            v = V[s]
            q_values = np.zeros(n_actions)
            for a in range(n_actions):
                # Calculate expected value for each action
                for prob, next_state, reward, done in env.unwrapped.P[s][a]:
                    q_values[a] += prob * (reward + gamma * V[next_state] * (not done))
            V[s] = np.max(q_values)
            delta = max(delta, abs(v - V[s]))
        if delta < theta:
            break

    # Get policy from the optimal Value Function
    for s in range(n_states):
        q_values = np.zeros(n_actions)
        for a in range(n_actions):
            for prob, next_state, reward, done in env.unwrapped.P[s][a]:
                q_values[a] += prob * (reward + gamma * V[next_state] * (not done))
        policy[s] = np.argmax(q_values)

    return V, policy

def print_policy(policy_array):
    action_map = {0: '←', 1: '↓', 2: '→', 3: '↑'}
    policy_grid = []

    print("\nFinal Learned Policy:")
    for state in range(16):
        # 5, 7, 11, 12 are Holes; 15 is the Goal
        if state in [5, 7, 11, 12, 15]:
            policy_grid.append('H/G')
        else:
            policy_grid.append(action_map[policy_array[state]])

    for i in range(0, 16, 4):
        print("".join([act.ljust(4) for act in policy_grid[i:i+4]]))

V_default, policy_default = value_iteration(env, gamma=0.99)

print("\nLearned Value Function (V):")
print(np.round(V_default.reshape(4, 4), 3))
print_policy(policy_default)


Learned Value Function (V):
[[0.542 0.499 0.471 0.457]
 [0.558 0.    0.358 0.   ]
 [0.592 0.643 0.615 0.   ]
 [0.    0.742 0.863 0.   ]]

Final Learned Policy:
←   ↑   ↑   ↑   
←   H/G ←   H/G 
↑   ↓   ←   H/G 
H/G →   ↓   H/G 


### Discussion
* This setup is an MDP because it fully satisfies the Markov property. The probability of transitioning to a new state and receiving a reward is fully dependent on the agen'ts current state and chosen action.

* The learned policy guides the agent along the safest route to the end goal, avoiding decisions that would put the agent in a dangerous situation if it were to slip. It hugs the edge of the lake so that it can guarantee it only moves along the safe boundary, until it is forced to move away.

### Discount Factor Experiment: Lowering Gamma to 0.1

In [2]:
low_gamma = 0.1

V_low_gamma, policy_low_gamma = value_iteration(env, gamma=low_gamma)

print(f"\nLearned Value Function (V) with Gamma = {low_gamma}:")
print(np.round(V_low_gamma.reshape(4, 4), 5))
print_policy(policy_low_gamma)


Learned Value Function (V) with Gamma = 0.1:
[[0.0000e+00 0.0000e+00 1.0000e-05 0.0000e+00]
 [0.0000e+00 0.0000e+00 3.9000e-04 0.0000e+00]
 [3.0000e-05 7.8000e-04 1.1550e-02 0.0000e+00]
 [0.0000e+00 1.1930e-02 3.4524e-01 0.0000e+00]]

Final Learned Policy:
↓   ↑   →   ↑   
←   H/G ←   H/G 
↑   ↓   ←   H/G 
H/G →   ↓   H/G 


* We see the learned policy takes a more dangerous but direct approach to the end goal, demonstrating the agents increased preference for immediate rewards.